In [1]:
import polars as pl
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
import gc

DATA_DIR = Path('../data/processed')
WIDE = DATA_DIR / 'wide'
K = 50

In [2]:
trainA = pl.read_parquet(WIDE / 'trainA.parquet')
valA   = pl.read_parquet(WIDE / 'valA.parquet')
testA  = pl.read_parquet(WIDE / 'testA.parquet')
trainB = pl.read_parquet(WIDE / 'trainB.parquet')
valB   = pl.read_parquet(WIDE / 'valB.parquet')
testB  = pl.read_parquet(WIDE / 'testB.parquet')

for name, df in [('trainA', trainA), ('valA', valA), ('testA', testA),
                 ('trainB', trainB), ('valB', valB), ('testB', testB)]:
    print(f'{name}: {df.shape}')

trainA: (20437223, 34)
valA: (3271268, 34)
testA: (2848377, 34)
trainB: (19281955, 33)
valB: (3078536, 33)
testB: (2667885, 33)


In [3]:
adFeatureRaw = pl.read_parquet(DATA_DIR / 'ad_feature.parquet')
adFeatureRaw = adFeatureRaw.filter(pl.col('price') <= 500_000)

uniqueCates = sorted(adFeatureRaw['cate_id'].unique().to_list())
cateMap = {orig: enc for enc, orig in enumerate(uniqueCates)}

print(f'Cate mapping: {len(cateMap):,} entries')
print(f'trainA.cate_id range: [{trainA["cate_id"].min()}, {trainA["cate_id"].max()}]')
print(f'Mapping range:        [0, {len(cateMap)-1}]')

Cate mapping: 6,769 entries
trainA.cate_id range: [0, 6768]
Mapping range:        [0, 6768]


In [4]:
cateMapDf = pl.DataFrame({
    'cate': uniqueCates,
    'cate_encoded': list(range(len(uniqueCates))),
})

validStart = int(datetime(2017, 4, 1).timestamp())
rawSampleMax = int(trainA['time_stamp'].max())

print('Loading and filtering behavior_log... (~1-2 min)')
behaviorLog = (
    pl.scan_parquet(DATA_DIR / 'behavior_log.parquet')
    .filter(
        (pl.col('time_stamp') >= validStart) &
        (pl.col('time_stamp') <= rawSampleMax)
    )
    .select(['user', 'time_stamp', 'cate'])
    .collect()
)
print(f'After time filter: {behaviorLog.shape}')

print('Encoding cate (inner join with mapping)...')
behaviorLog = (
    behaviorLog
    .join(cateMapDf, on='cate', how='inner')
    .drop('cate')
    .rename({'cate_encoded': 'cate'})
    .with_columns(pl.col('cate').cast(pl.UInt32))
)
print(f'After cate encoding: {behaviorLog.shape}')

print('Sorting by (user, time_stamp)...')
behaviorLog = behaviorLog.sort(['user', 'time_stamp'])
print('Done.')

Loading and filtering behavior_log... (~1-2 min)
After time filter: (657502835, 3)
Encoding cate (inner join with mapping)...
After cate encoding: (629320634, 3)
Sorting by (user, time_stamp)...
Done.


In [5]:
print('Converting to numpy arrays...')
user_arr = behaviorLog['user'].to_numpy()
time_arr = behaviorLog['time_stamp'].to_numpy()
cate_arr = behaviorLog['cate'].to_numpy().astype(np.int16)
print(f'Arrays: {len(user_arr):,} entries')

# Free polars frame to save memory
del behaviorLog
gc.collect()

print('Building per-user index ranges...')
unique_users, user_starts = np.unique(user_arr, return_index=True)
user_ends = np.append(user_starts[1:], len(user_arr))

user_to_range = {int(u): (int(s), int(e)) 
                 for u, s, e in zip(unique_users, user_starts, user_ends)}
print(f'{len(user_to_range):,} users with behavior history')

# Free user_arr (no longer needed)
del user_arr
gc.collect()

Converting to numpy arrays...
Arrays: 629,320,634 entries
Building per-user index ranges...
1,130,232 users with behavior history


0

In [6]:
def buildSequences(impressionsDf, user_to_range, time_arr, cate_arr, K=50):
    """For each impression, look up the user's most recent K behaviors before
    the impression timestamp. Returns a (n, K) int16 array; padding=0 at the front."""
    user_imp = impressionsDf['user'].to_numpy()
    time_imp = impressionsDf['time_stamp'].to_numpy()
    
    n = len(user_imp)
    out = np.zeros((n, K), dtype=np.int16)
    
    # Group impressions by user (sort by user, stable)
    sort_idx = np.argsort(user_imp, kind='stable')
    sorted_users = user_imp[sort_idx]
    sorted_times = time_imp[sort_idx]
    
    # Find ranges for each user in the sorted impressions
    unique_users_imp, imp_starts = np.unique(sorted_users, return_index=True)
    imp_ends = np.append(imp_starts[1:], len(sorted_users))
    
    for i, user_id in enumerate(unique_users_imp):
        user_id = int(user_id)
        if user_id not in user_to_range:
            continue  # all zeros (no behavior history)
        
        b_start, b_end = user_to_range[user_id]
        user_times = time_arr[b_start:b_end]
        user_cates = cate_arr[b_start:b_end]
        
        # Pad cates with K zeros at front for clean gathering
        padded = np.concatenate([np.zeros(K, dtype=np.int16), user_cates])
        
        # This user's impressions (vectorized lookup)
        i_start, i_end = imp_starts[i], imp_ends[i]
        query_times = sorted_times[i_start:i_end]
        
        # searchsorted: find first behavior >= query_time (strict less-than rule)
        idxs = np.searchsorted(user_times, query_times, side='left')
        # Gather K elements: padded[idx : idx+K] for each idx
        gather_idx = idxs[:, None] + np.arange(K)[None, :]
        sequences = padded[gather_idx]
        
        # Write back to original order using sort_idx
        out[sort_idx[i_start:i_end]] = sequences
    
    return out

print('Function defined.')

Function defined.


In [7]:
import time

print('Building sequences for trainA...')
t0 = time.time()
trainA_seqs = buildSequences(trainA, user_to_range, time_arr, cate_arr, K=K)
print(f'Done in {time.time()-t0:.1f} sec. Shape: {trainA_seqs.shape}')

# Sanity check on a few rows
print('\nFirst 3 rows:')
for i in range(3):
    seq = trainA_seqs[i]
    n_real = (seq != 0).sum()
    print(f'  Row {i}: user={trainA["user"][i]}, n_real_history={n_real}, '
          f'last 10: {seq[-10:].tolist()}')

# Distribution: how many impressions have full history vs cold-start
n_full = ((trainA_seqs != 0).sum(axis=1) == K).sum()
n_empty = ((trainA_seqs == 0).all(axis=1)).sum()
print(f'\nImpressions with full {K}-length history: {n_full:,} ({100*n_full/len(trainA_seqs):.1f}%)')
print(f'Impressions with NO history (all 0):       {n_empty:,} ({100*n_empty/len(trainA_seqs):.1f}%)')

Building sequences for trainA...
Done in 16.9 sec. Shape: (20437223, 50)

First 3 rows:
  Row 0: user=581737, n_real_history=50, last 10: [462, 462, 2653, 159, 3455, 3455, 5996, 3456, 1113, 1113]
  Row 1: user=399906, n_real_history=50, last 10: [2656, 2656, 2656, 2656, 2656, 2656, 2656, 2656, 2656, 2656]
  Row 2: user=628135, n_real_history=50, last 10: [2685, 2685, 2685, 2685, 2685, 1516, 5745, 1516, 1516, 1516]

Impressions with full 50-length history: 17,531,666 (85.8%)
Impressions with NO history (all 0):       395,909 (1.9%)


In [8]:
import time

# trainA: sequences already computed in Cell 7
print('=== trainA ===')
trainA = trainA.with_columns(
    behavior_seq=pl.Series('behavior_seq', list(trainA_seqs), dtype=pl.List(pl.Int16))
)
trainA.write_parquet(WIDE / 'trainA.parquet', compression='snappy')
sizeMb = (WIDE / 'trainA.parquet').stat().st_size / (1024**2)
print(f'  saved: {trainA.shape}, {sizeMb:.1f} MB')
del trainA_seqs
gc.collect()

# Process and save the other 5
for name in ['valA', 'testA', 'trainB', 'valB', 'testB']:
    print(f'\n=== {name} ===')
    df = globals()[name]
    t0 = time.time()
    seqs = buildSequences(df, user_to_range, time_arr, cate_arr, K=K)
    print(f'  built in {time.time()-t0:.1f} sec, shape {seqs.shape}')
    
    df = df.with_columns(
        behavior_seq=pl.Series('behavior_seq', list(seqs), dtype=pl.List(pl.Int16))
    )
    df.write_parquet(WIDE / f'{name}.parquet', compression='snappy')
    sizeMb = (WIDE / f'{name}.parquet').stat().st_size / (1024**2)
    print(f'  saved: {df.shape}, {sizeMb:.1f} MB')
    
    globals()[name] = df
    del seqs
    gc.collect()

print('\n=== ALL DONE ===')

# Final summary
total_size = sum((WIDE / f'{n}.parquet').stat().st_size for n in 
                 ['trainA', 'valA', 'testA', 'trainB', 'valB', 'testB']) / (1024**3)
print(f'Total disk: {total_size:.2f} GB')

=== trainA ===
  saved: (20437223, 35), 1836.9 MB

=== valA ===
  built in 5.3 sec, shape (3271268, 50)
  saved: (3271268, 35), 308.2 MB

=== testA ===
  built in 4.3 sec, shape (2848377, 50)
  saved: (2848377, 35), 265.6 MB

=== trainB ===
  built in 18.6 sec, shape (19281955, 50)
  saved: (19281955, 34), 1731.1 MB

=== valB ===
  built in 4.9 sec, shape (3078536, 50)
  saved: (3078536, 34), 289.9 MB

=== testB ===
  built in 5.9 sec, shape (2667885, 50)
  saved: (2667885, 34), 248.9 MB

=== ALL DONE ===
Total disk: 4.57 GB
